# Feature Selection

_Feature Engineering (Zheng & Casari)_

**Throw away features that do not help, so the model is cheaper, faster, and overfits less.**

More features is not more knowledge. Past a point, extra columns add noise, cost, and overfitting &mdash; the model fits quirks of the training sample instead of the real pattern. The goal of selection is a small subset that keeps the signal and drops the rest.

       The hard part is how you judge a feature. The three families are three answers, trading off speed against faithfulness to the real model:

---

This notebook is a practice scaffold for the **Feature Selection** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — scikit-learn

In [ ]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.feature_selection import SelectKBest, chi2, RFE
from sklearn.linear_model import LogisticRegression, Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score, StratifiedKFold

# The book demonstrates selection on high-dimensional text (bag-of-words / tf-idf).
# Same three families here on a bundled tabular dataset.
# Full notebooks: github.com/alicezheng/feature-engineering-book
X, y = load_breast_cancer(return_X_y=True)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

# ---------- (1) FILTER: score each feature alone, keep the top k ----------
# chi2 needs non-negative features (breast-cancer features are all >= 0).
filter_pipe = Pipeline([
    ("filter", SelectKBest(chi2, k=10)),   # univariate scores vs the target
    ("clf",    LogisticRegression(max_iter=5000)),
])
print("filter  (SelectKBest chi2, k=10):",
      round(cross_val_score(filter_pipe, X, y, cv=cv).mean(), 3))

# Inspect which features the filter kept (fit on all data only to LOOK):
skb = SelectKBest(chi2, k=10).fit(X, y)
print("  top-10 chi2 scores:", np.round(np.sort(skb.scores_)[::-1][:10], 1))

# ---------- (2) WRAPPER: recursive feature elimination around the model ----------
# RFE trains the model, drops the weakest feature, refits, repeats down to k.
wrapper_pipe = Pipeline([
    ("scale", StandardScaler()),
    ("rfe",   RFE(LogisticRegression(max_iter=5000),
                  n_features_to_select=10, step=1)),
    ("clf",   LogisticRegression(max_iter=5000)),
])
print("wrapper (RFE, k=10):",
      round(cross_val_score(wrapper_pipe, X, y, cv=cv).mean(), 3))

# ---------- (3) EMBEDDED: L1 (Lasso) sparsity selects during training ----------
# A larger alpha (lambda) zeroes out more coefficients -> fewer features kept.
lasso = Pipeline([("scale", StandardScaler()),
                  ("lasso", Lasso(alpha=0.05))]).fit(X, y)
coef = lasso.named_steps["lasso"].coef_
kept = np.flatnonzero(coef)              # non-zero weights == selected features
print("embedded (Lasso L1): kept %d of %d features" % (kept.size, X.shape[1]))
print("  selected feature indices:", kept.tolist())

## Visualize the data & results

_On real data, how many features do you actually need? A FILTER (mutual information with the target) ranks the features; we keep the top-k and watch accuracy. The curve plateaus fast: a handful of features gets almost all the accuracy._

In [ ]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold

X, y = load_breast_cancer(return_X_y=True)   # 569 rows, 30 features
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

# A FILTER ranks features by mutual information with the target; keep the top k.
# The selector lives INSIDE the pipeline, so it is re-fit on the training fold
# of every split -> no leakage of held-out rows into the ranking.
def mi(Xx, yy):
    return mutual_info_classif(Xx, yy, random_state=0)

ks = [1, 2, 3, 5, 8, 12, 20, 30]
accs = []
for k in ks:
    pipe = make_pipeline(
        StandardScaler(),
        SelectKBest(mi, k=k),                # filter: top-k by mutual information
        LogisticRegression(max_iter=5000),
    )
    accs.append(round(cross_val_score(pipe, X, y, cv=cv).mean(), 3))

for k, a in zip(ks, accs):
    print("k=%-3d  accuracy=%.3f" % (k, a))
# k=1 0.917 ... k=8 0.954 ... k=30 0.979  -> a few features get most of the accuracy.

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** You have a tf-idf matrix with 40,000 columns and need a quick first prune before training anything. Which family do you reach for, and what is the one number it cannot see?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Note the scale: 40,000 features means anything that trains a model per feature is too slow for a first pass. — _Wrappers train a model many times; that is impractical at 40,000 columns for a quick cut._
- Use a FILTER: score each column by chi-square or mutual information against the label and keep the top k. — _A filter costs one cheap statistic per feature and sorts &mdash; fast even at this width._
- Remember its blind spot: it scores each feature alone, so it cannot see INTERACTIONS between features. — _Two columns useless apart but predictive together would both be dropped by a filter._

**Answer:** Use a filter method (e.g. SelectKBest with chi2 or mutual information). It is the only family cheap enough for a fast first cut on 40,000 features. Its blind spot: it judges each feature on its own, so it cannot see feature interactions.

</details>

**Problem 2.** A colleague runs SelectKBest on the entire dataset, THEN splits into train and test, and is thrilled by the test accuracy. What did they do wrong, and how do you fix it?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Spot that the selector saw the test rows: the whole dataset, including test, was used to choose the features. — _Information from the held-out rows leaked into the feature choice, inflating the test score._
- Name it: this is data leakage. The test set is no longer truly unseen. — _Any step fit on data that includes the test set makes the score optimistic._
- Move the selector INSIDE a Pipeline and run it inside cross-validation, fitting on the training fold only. — _Then selection is re-done per fold on training data alone, so the held-out fold stays genuinely unseen._

**Answer:** They leaked: selecting on the whole dataset let the test rows influence which features were kept, so the test accuracy is optimistic. Fix: wrap the selector in a Pipeline and select inside cross-validation on the training fold only, never on the full data before the split.

</details>

**Problem 3.** You want feature selection to come "for free" as part of training a linear model, and you want unhelpful features driven to exactly zero. What do you use, and why does it produce exact zeros where ridge would not?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Choose an EMBEDDED method: a linear model with an L1 (Lasso) penalty. — _Embedded selection happens during the single fit; no separate scoring or subset search is needed._
- Recall the geometry: the L1 budget region is a diamond with corners on the axes; the loss contours often touch a corner. — _A corner is a point where some weights are exactly zero &mdash; those features are dropped._
- Contrast with L2 (ridge): its budget region is a smooth circle with no corners, so it shrinks weights but rarely zeroes them. — _Ridge keeps every feature (just smaller), so it does not select; Lasso does._

**Answer:** Use Lasso (an L1-penalized linear model) &mdash; selection is embedded in training. L1's diamond-shaped budget has corners on the axes, so the loss contours frequently touch at a point where some weights are exactly zero, dropping those features. L2 (ridge) has a smooth circular budget with no corners, so it only shrinks weights and never zeroes them.

</details>